# Dashboard Handoff Resource: Illegal Sand Mining (India)

This notebook is a compiled resource for the dashboard engineering team.
It converts existing analysis outputs into clean, dashboard-ready layer files and a manifest.

Primary sources used:
- spatial_analysis.ipynb
- geo.ipynb
- indiasandwatchEDA.ipynb

## 1. Handoff Scope

This notebook prepares:
- Mining point layer
- State-level mining intensity choropleth
- Water station and quality layers
- Air quality state layers (PM2.5, PM10)
- Crime state layer
- A single manifest JSON for frontend integration

It also includes a mapping of recommended dashboard modules and the exact files each module should use.

## 2. Dashboard Module Mapping

Use these modules in the dashboard:

1) Map Layers
- mining_points.geojson
- mining_state_intensity.geojson
- water_stations.geojson
- water_state_quality.geojson
- air_state_pm.geojson
- crime_state.geojson

2) Layer Controls
- Toggle groups: mining, water, air, crime
- Opacity slider per layer

3) Spatial Stats Panel
- Pull precomputed values from your analysis notebooks
- Optionally recompute in backend if interactive filters are needed

4) Model Insights Panel
- Feature importances and explanatory metrics from spatial_analysis.ipynb

5) Story/Narrative Panel
- Key findings and caveats from your final markdown summary

In [ ]:
from pathlib import Path
import json
import re
import unicodedata

import numpy as np
import pandas as pd
import geopandas as gpd


In [ ]:
# Project paths
ROOT = Path.cwd()
DATA = ROOT / 'data'
OUT = ROOT / 'dashboard_assets'
OUT.mkdir(parents=True, exist_ok=True)

print('Root:', ROOT)
print('Output:', OUT)


In [ ]:
# Name normalization helper to align state names across datasets
def norm_text(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r"[\./,&()'-]", ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return np.nan if s in {'', 'na', 'n/a', 'none', 'null'} else s

NAME_FIX = {
    'andaman nicobar': 'andaman and nicobar islands',
    'andaman and nicobar': 'andaman and nicobar islands',
    'dadra and nagar haveli and daman and diu': 'dadra and nagar haveli and daman diu',
    'nct of delhi': 'delhi',
    'orissa': 'odisha',
}


In [ ]:
# Parse mining coordinates from final_unified_geo_data_rows.csv
geo = pd.read_csv(ROOT / 'final_unified_geo_data_rows.csv')

def parse_raw_location(value):
    if pd.isna(value):
        return pd.Series([np.nan, np.nan])
    lines = [l.strip() for l in str(value).splitlines() if l.strip()]
    chosen = next((l for l in reversed(lines) if l.count(',') >= 2 and re.search(r'\d', l)), str(value))
    parts = [p.strip() for p in chosen.split(',')]
    if len(parts) < 3:
        return pd.Series([np.nan, np.nan])
    return pd.Series([parts[1], parts[2]])

geo[['latitude', 'longitude']] = geo['raw_location'].apply(parse_raw_location)
geo['latitude'] = pd.to_numeric(geo['latitude'], errors='coerce')
geo['longitude'] = pd.to_numeric(geo['longitude'], errors='coerce')
geo['state_n'] = geo['state'].apply(norm_text).replace(NAME_FIX)

mining_pts = geo.dropna(subset=['latitude', 'longitude']).copy()
mining_pts = mining_pts[mining_pts['latitude'].between(5, 40) & mining_pts['longitude'].between(65, 100)]

mining_gdf = gpd.GeoDataFrame(
    mining_pts,
    geometry=gpd.points_from_xy(mining_pts['longitude'], mining_pts['latitude']),
    crs='EPSG:4326'
)

print('Mining points:', len(mining_gdf))


In [ ]:
# Load states geometry
states = gpd.read_file(DATA / 'geo' / 'india_states.geojson')
nm_col = 'NAME_1' if 'NAME_1' in states.columns else next(c for c in states.columns if 'name' in c.lower())
states['state_n'] = states[nm_col].apply(norm_text).replace(NAME_FIX)

# Build state mining counts
state_mining = (
    geo.dropna(subset=['state_n'])
    .groupby('state_n', as_index=False)
    .size()
    .rename(columns={'size': 'mining_count'})
)

state_mining_gdf = states.merge(state_mining, on='state_n', how='left')
state_mining_gdf['mining_count'] = state_mining_gdf['mining_count'].fillna(0)

print('States with geometry:', len(states))


In [ ]:
# Water station point layer
gwq = pd.read_csv(DATA / 'env' / 'water' / 'WRIS Ground Water Quality Yearly.csv', low_memory=False)
gwq['lat'] = pd.to_numeric(gwq['Ground Water Station Latitude'], errors='coerce')
gwq['lon'] = pd.to_numeric(gwq['Ground Water Station Longitude'], errors='coerce')
gwq['state_n'] = gwq['State'].apply(norm_text).replace(NAME_FIX)

tds_col = next((c for c in gwq.columns if 'total dissolved solid' in c.lower()), None)
if tds_col is not None:
    gwq['tds'] = pd.to_numeric(gwq[tds_col], errors='coerce')
else:
    gwq['tds'] = np.nan

water_stn = gwq.dropna(subset=['lat', 'lon']).copy()
water_stn = water_stn[water_stn['lat'].between(5, 40) & water_stn['lon'].between(65, 100)]

water_stn_gdf = gpd.GeoDataFrame(
    water_stn[['state_n', 'lat', 'lon', 'tds']].copy(),
    geometry=gpd.points_from_xy(water_stn['lon'], water_stn['lat']),
    crs='EPSG:4326'
)

# State-level water quality summary
water_state = water_stn.groupby('state_n', as_index=False).agg(
    tds_median=('tds', 'median'),
    stations=('lat', 'count')
)
water_state_gdf = states.merge(water_state, on='state_n', how='left')

print('Water stations:', len(water_stn_gdf))


In [ ]:
# Air quality state layers
pm25 = pd.read_csv(DATA / 'env' / 'air' / 'PM2.5.csv')
pm10 = pd.read_csv(DATA / 'env' / 'air' / 'PM10.csv')

pm25['state_n'] = pm25['State'].apply(norm_text).replace(NAME_FIX)
pm10['state_n'] = pm10['State'].apply(norm_text).replace(NAME_FIX)

pm25_col = next((c for c in pm25.columns if 'annual average' in c.lower()), None)
pm10_col = next((c for c in pm10.columns if 'annual average' in c.lower()), None)

pm25['pm25'] = pd.to_numeric(pm25[pm25_col], errors='coerce') if pm25_col else np.nan
pm10['pm10'] = pd.to_numeric(pm10[pm10_col], errors='coerce') if pm10_col else np.nan

air_state = pm25.groupby('state_n', as_index=False)['pm25'].median().merge(
    pm10.groupby('state_n', as_index=False)['pm10'].median(),
    on='state_n',
    how='outer'
)

air_state_gdf = states.merge(air_state, on='state_n', how='left')

print('Air states with data:', air_state['state_n'].nunique())


In [ ]:
# Crime state layer
crime = pd.read_csv(DATA / 'crime' / 'state_wise_IPC' / 'swIPC_counts.csv')
crime['state_n'] = crime['State/UT'].apply(norm_text).replace(NAME_FIX)

crime_rate_col = next((c for c in crime.columns if 'rate' in c.lower() and 'cognizable' in c.lower()), None)
if crime_rate_col is None:
    crime_rate_col = next((c for c in crime.columns if 'rate' in c.lower()), None)

crime['crime_rate'] = pd.to_numeric(crime[crime_rate_col], errors='coerce') if crime_rate_col else np.nan
crime_state = crime[['state_n', 'crime_rate']].dropna(subset=['state_n']).copy()
crime_state_gdf = states.merge(crime_state, on='state_n', how='left')

print('Crime states with data:', crime_state['state_n'].nunique())


In [ ]:
# Export dashboard assets
paths = {}

paths['mining_points'] = str((OUT / 'mining_points.geojson').resolve())
mining_gdf.to_file(paths['mining_points'], driver='GeoJSON')

paths['mining_state_intensity'] = str((OUT / 'mining_state_intensity.geojson').resolve())
state_mining_gdf.to_file(paths['mining_state_intensity'], driver='GeoJSON')

paths['water_stations'] = str((OUT / 'water_stations.geojson').resolve())
water_stn_gdf.to_file(paths['water_stations'], driver='GeoJSON')

paths['water_state_quality'] = str((OUT / 'water_state_quality.geojson').resolve())
water_state_gdf.to_file(paths['water_state_quality'], driver='GeoJSON')

paths['air_state_pm'] = str((OUT / 'air_state_pm.geojson').resolve())
air_state_gdf.to_file(paths['air_state_pm'], driver='GeoJSON')

paths['crime_state'] = str((OUT / 'crime_state.geojson').resolve())
crime_state_gdf.to_file(paths['crime_state'], driver='GeoJSON')

print('Export complete.')
for key, value in paths.items():
    print(f'- {key}: {value}')


In [ ]:
# Write a manifest JSON for frontend/backend integration
manifest = {
    'project': 'Illegal Sand Mining India Dashboard',
    'generated_from': [
        'spatial_analysis.ipynb',
        'geo.ipynb',
        'indiasandwatchEDA.ipynb'
    ],
    'layers': [
        {'id': 'mining_points', 'type': 'point', 'group': 'mining', 'path': 'dashboard_assets/mining_points.geojson'},
        {'id': 'mining_state_intensity', 'type': 'polygon', 'group': 'mining', 'path': 'dashboard_assets/mining_state_intensity.geojson'},
        {'id': 'water_stations', 'type': 'point', 'group': 'water', 'path': 'dashboard_assets/water_stations.geojson'},
        {'id': 'water_state_quality', 'type': 'polygon', 'group': 'water', 'path': 'dashboard_assets/water_state_quality.geojson'},
        {'id': 'air_state_pm', 'type': 'polygon', 'group': 'air', 'path': 'dashboard_assets/air_state_pm.geojson'},
        {'id': 'crime_state', 'type': 'polygon', 'group': 'crime', 'path': 'dashboard_assets/crime_state.geojson'}
    ],
    'recommended_map_controls': {
        'layer_toggle': True,
        'opacity_slider': True,
        'legend_toggle': True,
        'attribute_popup': True
    },
    'notes': [
        'Marine layer is not generated in this notebook because dedicated marine geospatial source is not present.',
        'Spatial statistics and models are available in spatial_analysis.ipynb; frontend should treat them as analytical overlays or panel metrics.'
    ]
}

manifest_path = OUT / 'dashboard_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print('Manifest written:', manifest_path.resolve())


## 3. Dashboard Build Checklist (for engineering handoff)

1) Ingest dashboard_assets/dashboard_manifest.json
2) Render group toggles: mining, water, air, crime
3) Add independent opacity per visible layer
4) Add click popups (state_n, key metric fields)
5) Add stats panel pulling values from spatial_analysis.ipynb outputs
6) Add caveats block: no temporal causality claims

Suggested stack:
- Python backend: FastAPI or Flask for serving assets and model metrics
- Frontend map: Mapbox GL JS / deck.gl / Leaflet
- Charts: Plotly or ECharts